In [2]:
from datasets import load_dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
import torch

# 1. Load and prepare the dataset
dataset = load_dataset("csv", data_files="data.csv")

# Split the dataset into train and validation sets
train_dataset = dataset["train"]

# 2. Load the GPT-2 model and tokenizer
model_name = "gpt2"  # You can use "gpt2-medium" or "gpt2-large" for more capacity
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

# GPT-2 tokenizer does not include padding by default
tokenizer.pad_token = tokenizer.eos_token

# 3. Preprocess the dataset
def preprocess_function(examples):
    # Combine question, context, and answer into a single string
    inputs = [f"question: {q} context: {c} answer: {a}" for q, c, a in zip(examples['question'], examples['context'], examples['answer'])]
    
    # Tokenize the combined string
    model_inputs = tokenizer(inputs, max_length=512, padding="max_length", truncation=True)
    
    # The labels should be the same as the input IDs
    # GPT expects labels for causal language modeling
    model_inputs["labels"] = model_inputs["input_ids"].copy()

    return model_inputs

# Apply preprocessing
tokenized_dataset = train_dataset.map(preprocess_function, batched=True)

# 4. Set up training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=10_000,
    save_total_limit=2,
    logging_dir='./logs',
    logging_steps=500,
    evaluation_strategy="no",  # Disable evaluation
)

# 5. Initialize the Trainer
trainer = Trainer(
    model=model,                         # the model to be trained
    args=training_args,                  # training arguments
    train_dataset=tokenized_dataset,     # training dataset
    tokenizer=tokenizer,                 # tokenizer for text processing
)

# 6. Train the model
trainer.train()

# 7. Save the fine-tuned model
model.save_pretrained('./fine_tuned_gpt2')
tokenizer.save_pretrained('./fine_tuned_gpt2')

# 8. Optionally, evaluate the model
# Split your dataset into training and evaluation
eval_dataset = tokenized_dataset.select(range(len(tokenized_dataset) // 2))  # Example split

# Evaluate using the eval_dataset
results = trainer.evaluate(eval_dataset=eval_dataset)

# Print evaluation results
print(results)


Map: 100%|██████████| 4/4 [00:00<00:00, 166.35 examples/s]
                                     
100%|██████████| 3/3 [00:29<00:00,  9.85s/it]


{'train_runtime': 29.5322, 'train_samples_per_second': 0.406, 'train_steps_per_second': 0.102, 'train_loss': 4.233160654703776, 'epoch': 3.0}


100%|██████████| 1/1 [00:00<00:00, 166.58it/s]

{'eval_loss': 0.6859872937202454, 'eval_runtime': 1.7519, 'eval_samples_per_second': 1.142, 'eval_steps_per_second': 0.571, 'epoch': 3.0}


In [12]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# 1. Load the fine-tuned model and tokenizer
model_name_or_path = "./fine_tuned_gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name_or_path)
model = GPT2LMHeadModel.from_pretrained(model_name_or_path)

# Ensure the pad token is set
tokenizer.pad_token = tokenizer.eos_token

# 2. Prepare the input
def prepare_input(question, context):
    """
    Prepare the input string in the same format as used during training.
    """
    input_text = f"question: {question} context: {context} answer:"
    return input_text

question = "what supervised learninng?"
context = "Supervised learning is a type of machine learning where the model is trained using labeled data. The goal is to teach the model to make predictions based on known outcomes."
input_text = prepare_input(question, context)

# Tokenize the input
input_ids = tokenizer(input_text, return_tensors="pt").input_ids

# 3. Generate the output
output_ids = model.generate(
    input_ids=input_ids,
    max_length=200,  # Adjust based on the expected answer length
    num_return_sequences=1,  # Number of outputs to generate
    temperature=0.7,  # Controls creativity
    top_k=50,  # Filters to top K likely tokens
    top_p=0.95,  # Nucleus sampling
    pad_token_id=tokenizer.eos_token_id,  # Ensure the model uses the correct pad token
)

# Decode the generated output
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

# 4. Extract the answer
# The answer follows "answer:" in the generated text
answer = output_text.split("answer:")[-1].strip()

# 5. Print the result
print("Generated Answer:", answer)


Generated Answer: supervised learning is a type of machine learning where the model is trained using labeled data. The goal is to teach the model to make predictions based on known outcomes.
